# Notebook 3 — FakeHealth · LLM Zero-Shot & Few-Shot Evaluation
## Comparing GPT-4o / Claude / Mistral against the TF-IDF Baseline (Table 1)

**Project:** A Multi-Source Benchmark for CVD Health Misinformation Detection  
**Dataset:** FakeHealth — HealthStory — Dai, Sun & Wang (ICWSM 2020)  
**Prerequisites:** Run NB2 first to generate `healthstory_test_set.csv` and `Table1_baseline_results.csv`

---
### What does this notebook do?
1. Load the shared test set produced by NB2 (`healthstory_test_set.csv`)
2. Define a unified evaluation harness (same metrics as NB2)
3. Evaluate LLMs in **zero-shot** mode (no examples in the prompt)
4. Evaluate LLMs in **few-shot** mode (5 balanced examples in the prompt)
5. Compare all results against the NB2 baseline in a unified **Table 2**
6. Analyse error patterns: where do LLMs fail vs. the TF-IDF baseline?
7. Save all predictions for downstream cross-dataset experiments (NB4)

---
### Supported LLMs
| Provider | Model | API key env var |
|---|---|---|
| OpenAI | `gpt-4o` | `OPENAI_API_KEY` |
| Anthropic | `claude-3-5-sonnet-20241022` | `ANTHROPIC_API_KEY` |
| Mistral AI | `mistral-large-latest` | `MISTRAL_API_KEY` |

Set the keys you have as environment variables. Models with a missing key are skipped automatically.

---
## Section 1 — Installation & Imports

In [1]:
# Uncomment to install if needed:
# !pip install openai anthropic mistralai pandas numpy matplotlib seaborn scikit-learn tqdm joblib

import os
import time
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Imports OK')

Imports OK


---
## Section 2 — Load Shared Test Set from NB2

We use the **exact same test set** as NB2 so that all results are directly comparable.

In [ ]:
# ── Path to the test set exported by NB2 ─────────────────────────────
TEST_SET_PATH = 'healthstory_test_set.csv'
TABLE1_PATH   = 'Table1_baseline_results.csv'
# ─────────────────────────────────────────────────────────────────────

if not os.path.exists(TEST_SET_PATH):
    raise FileNotFoundError(
        f'Test set not found: {TEST_SET_PATH}\n'
        'Please run NB2 first to generate this file.'
    )

test_df = pd.read_csv(TEST_SET_PATH, encoding='utf-8')
print(f'Test set loaded: {len(test_df)} articles')
print(f"   Fake (0) : {(test_df['label']==0).sum()} | Real (1) : {(test_df['label']==1).sum()}")
print()

# Load NB2 baseline results if available
if os.path.exists(TABLE1_PATH):
    table1 = pd.read_csv(TABLE1_PATH)
    print(' Table 1 (baseline) loaded:')
    print(table1[['Model', 'Accuracy', 'F1_macro', 'F1_Fake', 'F1_Real']].to_string(index=False))
else:
    print(f' Baseline table not found at {TABLE1_PATH} — will create Table 2 without it.')
    table1 = None

Test set loaded: 321 articles
   Fake (0) : 90 | Real (1) : 231

✅ Table 1 (baseline) loaded:
              Model  Accuracy  F1_macro  F1_Fake  F1_Real
Logistic Regression     0.607     0.545    0.376    0.714
        Naive Bayes     0.713     0.502    0.179    0.826
          LinearSVC     0.626     0.564    0.400    0.729


---
## Section 3 — LLM Client Setup

Each LLM is wrapped in a simple class with a `predict(text)` method that returns `0` (Fake) or `1` (Real).

**Design principle:** All models receive the same prompt template, so differences in score are due to the model, not the prompt.

In [3]:
# ── Prompt templates ──────────────────────────────────────────────────

SYSTEM_PROMPT = """You are an expert fact-checker specialising in health and medical news.
Your task is to classify a news article as either REAL or FAKE health news.

Definitions:
- REAL: The article contains accurate, evidence-based health information.
- FAKE: The article contains misinformation, exaggeration, or unsupported health claims.

Respond with ONLY one word: REAL or FAKE. No explanation."""


def build_zero_shot_prompt(text: str) -> str:
    """Zero-shot: just the article, no examples."""
    return f"Article:\n{text[:2000]}\n\nClassification:"


def build_few_shot_prompt(text: str, examples: list[dict]) -> str:
    """
    Few-shot: prepend N balanced examples before the target article.
    Each example is a dict with keys: 'text', 'label' (0=Fake, 1=Real).
    """
    label_map = {0: 'FAKE', 1: 'REAL'}
    shots = []
    for ex in examples:
        shots.append(
            f"Article:\n{str(ex['text'])[:500]}\n"
            f"Classification: {label_map[int(ex['label'])]}"
        )
    shots_str = '\n\n'.join(shots)
    return f"{shots_str}\n\nArticle:\n{text[:2000]}\n\nClassification:"


def parse_label(response: str) -> int | None:
    """
    Parse LLM response into binary label.
    Returns 1 (Real), 0 (Fake), or None if unparseable.
    """
    r = response.strip().upper()
    if 'FAKE' in r:
        return 0
    if 'REAL' in r:
        return 1
    return None  # unparseable


print(' Prompt templates defined')

 Prompt templates defined


In [ ]:
# ── LLM wrappers ─────────────────────────────────────────────────────

class OpenAIClassifier:
    """
    Wraps the OpenAI chat completion API.
    Requires: OPENAI_API_KEY environment variable.
    """
    def __init__(self, model: str = 'gpt-4o', max_retries: int = 3, delay: float = 1.0):
        import openai
        self.client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])
        self.model = model
        self.max_retries = max_retries
        self.delay = delay

    def predict_raw(self, user_content: str) -> str:
        for attempt in range(self.max_retries):
            try:
                resp = self.client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {'role': 'system', 'content': SYSTEM_PROMPT},
                        {'role': 'user',   'content': user_content},
                    ],
                    max_tokens=5,
                    temperature=0.0,
                )
                return resp.choices[0].message.content.strip()
            except Exception as e:
                if attempt < self.max_retries - 1:
                    time.sleep(self.delay * (attempt + 1))
                else:
                    print(f'    OpenAI error after {self.max_retries} retries: {e}')
                    return 'ERROR'


class AnthropicClassifier:
    """
    Wraps the Anthropic Messages API.
    Requires: ANTHROPIC_API_KEY environment variable.
    """
    def __init__(self, model: str = 'claude-3-5-sonnet-20241022', max_retries: int = 3, delay: float = 1.0):
        import anthropic
        self.client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
        self.model = model
        self.max_retries = max_retries
        self.delay = delay

    def predict_raw(self, user_content: str) -> str:
        for attempt in range(self.max_retries):
            try:
                resp = self.client.messages.create(
                    model=self.model,
                    system=SYSTEM_PROMPT,
                    messages=[{'role': 'user', 'content': user_content}],
                    max_tokens=5,
                )
                return resp.content[0].text.strip()
            except Exception as e:
                if attempt < self.max_retries - 1:
                    time.sleep(self.delay * (attempt + 1))
                else:
                    print(f'    Anthropic error after {self.max_retries} retries: {e}')
                    return 'ERROR'


class MistralClassifier:
    """
    Wraps the Mistral AI chat API.
    Requires: MISTRAL_API_KEY environment variable.
    """
    def __init__(self, model: str = 'mistral-large-latest', max_retries: int = 3, delay: float = 1.0):
        from mistralai import Mistral
        self.client = Mistral(api_key=os.environ['MISTRAL_API_KEY'])
        self.model = model
        self.max_retries = max_retries
        self.delay = delay

    def predict_raw(self, user_content: str) -> str:
        for attempt in range(self.max_retries):
            try:
                resp = self.client.chat.complete(
                    model=self.model,
                    messages=[
                        {'role': 'system', 'content': SYSTEM_PROMPT},
                        {'role': 'user',   'content': user_content},
                    ],
                    max_tokens=5,
                    temperature=0.0,
                )
                return resp.choices[0].message.content.strip()
            except Exception as e:
                if attempt < self.max_retries - 1:
                    time.sleep(self.delay * (attempt + 1))
                else:
                    print(f'    Mistral error after {self.max_retries} retries: {e}')
                    return 'ERROR'


print('✅ LLM wrapper classes defined')

In [ ]:
# ── Instantiate available LLMs ────────────────────────────────────────
# A model is skipped if its API key is not set in the environment.

llm_registry = {
    'GPT-4o':   ('OPENAI_API_KEY',    lambda: OpenAIClassifier(model='gpt-4o')),
    'Claude-3.5-Sonnet': ('ANTHROPIC_API_KEY', lambda: AnthropicClassifier(model='claude-3-5-sonnet-20241022')),
    'Mistral-Large':     ('MISTRAL_API_KEY',   lambda: MistralClassifier(model='mistral-large-latest')),
}

active_llms = {}
for name, (env_var, factory) in llm_registry.items():
    if os.environ.get(env_var):
        try:
            active_llms[name] = factory()
            print(f'  {name} — ready ({env_var} found)')
        except Exception as e:
            print(f'   {name} — init failed: {e}')
    else:
        print(f'  Warning---> {name} — skipped ({env_var} not set)')

if not active_llms:
    print()
    print('No LLMs active. Set at least one API key to run this notebook.')
    print('Example (bash):  export OPENAI_API_KEY=sk-...')

---
## Section 4 — Build Few-Shot Example Pool

We select `k` balanced examples **from the training portion only** — the test set must remain unseen.

The training portion is all articles NOT in `healthstory_test_set.csv`. If the full dataset is not available, we fall back to selecting from the test set header (not ideal, but workable for prototyping).

In [ ]:
K_SHOTS = 5  # total few-shot examples (K_SHOTS // 2 Fake + K_SHOTS // 2 Real)

# ── Option A: load training CSV if available ──────────────────────────
train_csv_candidates = [
    '../dataset/healthstory_baseline_ready.csv',
    'HealthStory_clean.csv',
]
train_csv_path = next((p for p in train_csv_candidates if os.path.exists(p)), None)

if train_csv_path:
    full_df = pd.read_csv(train_csv_path, encoding='utf-8')

    # Reconstruct the same 80/20 split as NB2
    from sklearn.model_selection import train_test_split
    has_body  = 'body'  in full_df.columns
    has_title = 'title' in full_df.columns
    if has_body and has_title:
        full_df['text'] = full_df['title'].fillna('') + ' ' + full_df['body'].fillna('')
    elif has_body:
        full_df['text'] = full_df['body'].fillna('')

    full_df = full_df.dropna(subset=['label', 'text'])
    full_df = full_df[full_df['text'].str.strip().astype(bool)]

    train_idx, _ = train_test_split(
        np.arange(len(full_df)),
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=full_df['label']
    )
    train_pool = full_df.iloc[train_idx][['text', 'label']].reset_index(drop=True)
    print(f'✅ Train pool built from {train_csv_path}: {len(train_pool)} articles')

else:
    # Fallback: use test set (sub-optimal but workable)
    train_pool = test_df.copy()
    print('⚠️  Full dataset CSV not found — using test set as fallback for few-shot examples.')
    print('   For rigorous evaluation, run NB1 and NB2 first to generate the clean CSV.')

# Select K_SHOTS balanced examples
k_per_class = K_SHOTS // 2
few_shot_examples = pd.concat([
    train_pool[train_pool['label'] == 0].sample(n=k_per_class, random_state=RANDOM_STATE),
    train_pool[train_pool['label'] == 1].sample(n=k_per_class, random_state=RANDOM_STATE),
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Handle odd K_SHOTS — add one more Fake example
if K_SHOTS % 2 == 1:
    extra = train_pool[train_pool['label'] == 0].sample(n=1, random_state=RANDOM_STATE + 1)
    few_shot_examples = pd.concat([few_shot_examples, extra]).reset_index(drop=True)

few_shot_list = few_shot_examples.to_dict(orient='records')

print(f'\n── {K_SHOTS}-shot examples ──')
for ex in few_shot_list:
    label_str = 'FAKE' if ex['label'] == 0 else 'REAL'
    print(f'  [{label_str}] {str(ex["text"])[:80]}...')

---
## Section 5 — Evaluation Harness

A single function runs any LLM on the full test set and returns a result dict compatible with NB2's format.

In [ ]:
def evaluate_llm(
    llm,
    test_df: pd.DataFrame,
    mode: str = 'zero_shot',          # 'zero_shot' or 'few_shot'
    few_shot_examples: list = None,
    rate_limit_delay: float = 0.5,    # seconds between API calls
    cache_path: str = None,           # if set, save/load predictions here
) -> dict:
    """
    Evaluate an LLM classifier on the full test set.
    Returns a results dict matching NB2 test_results format.

    Parameters
    ----------
    llm              : LLM wrapper (OpenAIClassifier, AnthropicClassifier, ...)
    test_df          : DataFrame with columns ['text', 'label']
    mode             : 'zero_shot' or 'few_shot'
    few_shot_examples: list of dicts [{'text': ..., 'label': 0/1}, ...]
    rate_limit_delay : pause between API calls (seconds)
    cache_path       : optional CSV path to cache / resume predictions
    """
    assert mode in ('zero_shot', 'few_shot'), "mode must be 'zero_shot' or 'few_shot'"

    texts  = test_df['text'].tolist()
    labels = test_df['label'].tolist()

    # ── Resume from cache ──────────────────────────────────────────────
    if cache_path and os.path.exists(cache_path):
        cached = pd.read_csv(cache_path)
        preds_raw = cached['pred_raw'].tolist()
        preds     = cached['pred'].tolist()
        print(f'  📂 Loaded {len(preds)} cached predictions from {cache_path}')
    else:
        preds_raw = []
        preds     = []

    start_idx = len(preds)

    # ── Main inference loop ───────────────────────────────────────────
    for i, text in enumerate(tqdm(texts[start_idx:], initial=start_idx, total=len(texts),
                                   desc=f'{mode}')):
        if mode == 'zero_shot':
            prompt = build_zero_shot_prompt(text)
        else:
            prompt = build_few_shot_prompt(text, few_shot_examples)

        raw  = llm.predict_raw(prompt)
        pred = parse_label(raw)

        # If unparseable, default to majority class (Real=1)
        if pred is None:
            pred = 1
            print(f'  ⚠️  Unparseable response at index {start_idx + i}: "{raw}" → defaulted to Real')

        preds_raw.append(raw)
        preds.append(pred)

        time.sleep(rate_limit_delay)

        # ── Incremental cache ──────────────────────────────────────────
        if cache_path and (i + 1) % 50 == 0:
            cache_df = pd.DataFrame({'text': texts[:len(preds)], 'label': labels[:len(preds)],
                                     'pred_raw': preds_raw, 'pred': preds})
            cache_df.to_csv(cache_path, index=False)

    # ── Final cache save ──────────────────────────────────────────────
    if cache_path:
        cache_df = pd.DataFrame({'text': texts, 'label': labels,
                                  'pred_raw': preds_raw, 'pred': preds})
        cache_df.to_csv(cache_path, index=False)

    # ── Metrics ───────────────────────────────────────────────────────
    y_true = np.array(labels)
    y_pred = np.array(preds)

    results = {
        'y_pred'     : y_pred,
        'y_true'     : y_true,
        'preds_raw'  : preds_raw,
        'accuracy'   : accuracy_score(y_true, y_pred),
        'prec_macro' : precision_score(y_true, y_pred, average='macro', zero_division=0),
        'rec_macro'  : recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro'   : f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_fake'    : f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        'f1_real'    : f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'n_unparseable': sum(1 for r in preds_raw if parse_label(r) is None),
    }

    return results


print('✅ Evaluation harness defined')

---
## Section 6 — Run Zero-Shot Evaluation

Each active LLM is evaluated on the full test set with no examples in the prompt.  
Predictions are cached to CSV so the notebook can be safely interrupted and resumed.

In [ ]:
os.makedirs('predictions', exist_ok=True)

zero_shot_results = {}

for name, llm in active_llms.items():
    safe_name  = name.lower().replace(' ', '_').replace('-', '_').replace('.', '_')
    cache_path = f'predictions/{safe_name}_zero_shot.csv'

    print(f'\n── Zero-shot: {name} ──')
    res = evaluate_llm(
        llm,
        test_df,
        mode='zero_shot',
        cache_path=cache_path,
        rate_limit_delay=0.5,
    )
    zero_shot_results[f'{name} (zero-shot)'] = res

    print(f'  Accuracy  : {res["accuracy"]:.4f}')
    print(f'  F1 macro  : {res["f1_macro"]:.4f}')
    print(f'  F1 Fake   : {res["f1_fake"]:.4f}')
    print(f'  F1 Real   : {res["f1_real"]:.4f}')
    if res['n_unparseable'] > 0:
        print(f'  ⚠️  Unparseable responses: {res["n_unparseable"]}')

    print()
    print(classification_report(
        res['y_true'], res['y_pred'],
        target_names=['Fake', 'Real']
    ))

---
## Section 7 — Run Few-Shot Evaluation

Same LLMs, same test set, but with 5 balanced examples prepended to each prompt.

In [ ]:
few_shot_results = {}

for name, llm in active_llms.items():
    safe_name  = name.lower().replace(' ', '_').replace('-', '_').replace('.', '_')
    cache_path = f'predictions/{safe_name}_few_shot_{K_SHOTS}.csv'

    print(f'\n── {K_SHOTS}-shot: {name} ──')
    res = evaluate_llm(
        llm,
        test_df,
        mode='few_shot',
        few_shot_examples=few_shot_list,
        cache_path=cache_path,
        rate_limit_delay=0.5,
    )
    few_shot_results[f'{name} ({K_SHOTS}-shot)'] = res

    print(f'  Accuracy  : {res["accuracy"]:.4f}')
    print(f'  F1 macro  : {res["f1_macro"]:.4f}')
    print(f'  F1 Fake   : {res["f1_fake"]:.4f}')
    print(f'  F1 Real   : {res["f1_real"]:.4f}')

    print()
    print(classification_report(
        res['y_true'], res['y_pred'],
        target_names=['Fake', 'Real']
    ))

---
## Section 8 — Confusion Matrices

In [ ]:
all_llm_results = {**zero_shot_results, **few_shot_results}

if not all_llm_results:
    print('No LLM results to plot. Set at least one API key and re-run Sections 6–7.')
else:
    n = len(all_llm_results)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.5))
    if n == 1:
        axes = [axes]

    for ax, (name, res) in zip(axes, all_llm_results.items()):
        cm   = confusion_matrix(res['y_true'], res['y_pred'])
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fake', 'Real'])
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(f'{name}\nF1-macro = {res["f1_macro"]:.3f}',
                     fontsize=10, fontweight='bold')
        fn = cm[1][0]
        fp = cm[0][1]
        ax.set_xlabel(f'Predicted\n(FP={fp}, FN={fn})')

    plt.suptitle('Confusion Matrices — LLM Evaluation (HealthStory Test Set)',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('fig_llm_confusion_matrices.png', bbox_inches='tight', dpi=130)
    plt.show()
    print('💾 Saved: fig_llm_confusion_matrices.png')

---
## Section 9 — Zero-Shot vs Few-Shot Delta Analysis

How much do the few-shot examples help (or hurt) each model?

In [ ]:
delta_rows = []

for name in active_llms.keys():
    zs_key = f'{name} (zero-shot)'
    fs_key = f'{name} ({K_SHOTS}-shot)'

    if zs_key in zero_shot_results and fs_key in few_shot_results:
        zs = zero_shot_results[zs_key]
        fs = few_shot_results[fs_key]
        delta_rows.append({
            'Model'             : name,
            'F1_macro_ZS'       : round(zs['f1_macro'], 3),
            'F1_macro_FS'       : round(fs['f1_macro'], 3),
            'Delta_F1_macro'    : round(fs['f1_macro'] - zs['f1_macro'], 3),
            'F1_Fake_ZS'        : round(zs['f1_fake'], 3),
            'F1_Fake_FS'        : round(fs['f1_fake'], 3),
            'Delta_F1_Fake'     : round(fs['f1_fake'] - zs['f1_fake'], 3),
        })

if delta_rows:
    delta_df = pd.DataFrame(delta_rows)
    print('── Zero-Shot vs Few-Shot Delta ──')
    print(delta_df.to_string(index=False))
    print()
    print('Positive Δ = few-shot helps | Negative Δ = few-shot hurts')
else:
    print('No delta to compute (need both zero-shot and few-shot results).')

---
## Section 10 — Error Analysis: Where do LLMs fail?

We focus on **false negatives** (Fake articles classified as Real) — these are the most dangerous errors in a health misinformation context.

In [ ]:
# Use the best available LLM in zero-shot mode for detailed error analysis
if zero_shot_results:
    best_name = max(zero_shot_results, key=lambda k: zero_shot_results[k]['f1_macro'])
    best_res  = zero_shot_results[best_name]

    analysis_df = test_df.copy().reset_index(drop=True)
    analysis_df['y_true'] = best_res['y_true']
    analysis_df['y_pred'] = best_res['y_pred']
    analysis_df['correct'] = (analysis_df['y_true'] == analysis_df['y_pred'])

    errors = analysis_df[~analysis_df['correct']]
    fp = errors[(errors['y_true'] == 1) & (errors['y_pred'] == 0)]  # Real → Fake
    fn = errors[(errors['y_true'] == 0) & (errors['y_pred'] == 1)]  # Fake → Real

    print(f'Error analysis for: {best_name}')
    print(f'Total errors    : {len(errors)} / {len(analysis_df)} ({100*len(errors)/len(analysis_df):.1f}%)')
    print(f'False Positives (Real → Fake) : {len(fp)}')
    print(f'False Negatives (Fake → Real) : {len(fn)}  ← critical: misinformation missed')
    print()

    print('── Top 3 False Negatives (Fake articles predicted as Real) ──')
    print('   These contain misinformation but sound credible to the LLM')
    print()
    for i, (_, row) in enumerate(fn.head(3).iterrows()):
        txt = str(row.get('text', 'N/A'))[:300]
        print(f'  [{i+1}] {txt}...')
        print()

    print('── Top 3 False Positives (Real articles predicted as Fake) ──')
    print()
    for i, (_, row) in enumerate(fp.head(3).iterrows()):
        txt = str(row.get('text', 'N/A'))[:300]
        print(f'  [{i+1}] {txt}...')
        print()
else:
    print('No zero-shot results available for error analysis.')

---
## Section 11 — Comprehensive Comparison Chart

A grouped bar chart showing all models (baseline + LLMs) side by side on the four key metrics.

In [ ]:
# Build a unified results dict: baselines + LLMs
all_results_for_plot = {}

# Add NB2 baselines if available
if table1 is not None:
    for _, row in table1.iterrows():
        all_results_for_plot[row['Model'] + ' (baseline)'] = {
            'accuracy'   : row['Accuracy'],
            'f1_macro'   : row['F1_macro'],
            'f1_fake'    : row['F1_Fake'],
            'f1_real'    : row['F1_Real'],
        }

# Add LLM results
for name, res in all_llm_results.items():
    all_results_for_plot[name] = {
        'accuracy'  : res['accuracy'],
        'f1_macro'  : res['f1_macro'],
        'f1_fake'   : res['f1_fake'],
        'f1_real'   : res['f1_real'],
    }

if not all_results_for_plot:
    print('No results to plot.')
else:
    metrics = ['accuracy', 'f1_macro', 'f1_fake', 'f1_real']
    metric_labels = ['Accuracy', 'F1 (macro)', 'F1 (Fake)', 'F1 (Real)']
    model_names = list(all_results_for_plot.keys())

    x     = np.arange(len(metrics))
    width = 0.8 / max(len(model_names), 1)

    # Colour scheme: grey for baselines, blues/greens for LLMs
    palette = [
        '#95a5a6', '#7f8c8d', '#bdc3c7',   # baseline greys
        '#2980b9', '#27ae60', '#8e44ad',    # LLM blues/greens/purples
        '#e67e22', '#c0392b', '#1abc9c',
    ]

    fig, ax = plt.subplots(figsize=(13, 6))

    for i, (name, res) in enumerate(all_results_for_plot.items()):
        vals = [res[m] for m in metrics]
        color = palette[i % len(palette)]
        bars = ax.bar(x + i * width, vals, width, label=name, color=color, alpha=0.85, edgecolor='white')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=7)

    ax.set_xticks(x + width * (len(model_names) - 1) / 2)
    ax.set_xticklabels(metric_labels, fontsize=11)
    ax.set_ylim(0, 1.10)
    ax.set_ylabel('Score')
    ax.set_title('Baseline vs LLM Performance — HealthStory (FakeHealth)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right', ncol=2)
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)

    plt.tight_layout()
    plt.savefig('fig_llm_vs_baseline_comparison.png', bbox_inches='tight', dpi=130)
    plt.show()
    print('💾 Saved: fig_llm_vs_baseline_comparison.png')

---
## Section 12 — TABLE 2 (Final Report)

This is the main result table of the report, extending Table 1 with LLM results.

In [ ]:
table2_rows = []

# ── Baselines from NB2 ────────────────────────────────────────────────
if table1 is not None:
    for _, row in table1.iterrows():
        table2_rows.append({
            'Model'       : row['Model'],
            'Type'        : 'Baseline (TF-IDF)',
            'Accuracy'    : row['Accuracy'],
            'Precision'   : row['Precision'],
            'Recall'      : row['Recall'],
            'F1_macro'    : row['F1_macro'],
            'F1_Fake'     : row['F1_Fake'],
            'F1_Real'     : row['F1_Real'],
            'CV_F1'       : f"{row['CV_F1_mean']:.3f}±{row['CV_F1_std']:.3f}" if 'CV_F1_mean' in row else 'N/A',
        })

# ── LLM results ───────────────────────────────────────────────────────
for name, res in all_llm_results.items():
    mode = 'zero-shot' if 'zero-shot' in name else f'{K_SHOTS}-shot'
    table2_rows.append({
        'Model'       : name,
        'Type'        : f'LLM ({mode})',
        'Accuracy'    : round(res['accuracy'],   3),
        'Precision'   : round(res['prec_macro'],  3),
        'Recall'      : round(res['rec_macro'],   3),
        'F1_macro'    : round(res['f1_macro'],    3),
        'F1_Fake'     : round(res['f1_fake'],     3),
        'F1_Real'     : round(res['f1_real'],     3),
        'CV_F1'       : 'N/A',
    })

table2 = pd.DataFrame(table2_rows)

print('=' * 100)
print('TABLE 2 — Baseline + LLM Performance — HealthStory (FakeHealth)')
print('Features (baseline): TF-IDF unigrammes + bigrammes (max 10 000)')
print('LLM prompting: zero-shot and 5-shot with balanced examples')
print('Evaluation: same 20% test set across all models')
print('=' * 100)

header = (f'{"Model":<40} {"Type":<22} {"Acc":>7} {"Prec":>7} {"Recall":>7} '
          f'{"F1-mac":>7} {"F1-Fake":>8} {"F1-Real":>8} {"CV F1":>10}')
print(header)
print('-' * 100)

for _, row in table2.iterrows():
    print(
        f'{row["Model"]:<40} {row["Type"]:<22} '
        f'{row["Accuracy"]:>7.3f} {row["Precision"]:>7.3f} {row["Recall"]:>7.3f} '
        f'{row["F1_macro"]:>7.3f} {row["F1_Fake"]:>8.3f} {row["F1_Real"]:>8.3f} '
        f'{str(row["CV_F1"]):>10}'
    )

print('=' * 100)
print()
print('Notes:')
print('  • F1-macro = main metric (slight class imbalance ~55/45)')
print('  • F1-Fake  = most critical metric (ability to detect misinformation)')
print('  • CV F1    = 5-fold CV from NB2 (only available for TF-IDF baselines)')
print('  • LLM prompts use the same system message for all models')

table2.to_csv('Table2_llm_vs_baseline_results.csv', index=False)
print()
print('💾 Saved: Table2_llm_vs_baseline_results.csv')

---
## Section 13 — Save Predictions for NB4 (Cross-Dataset Generalisation)

We export predictions in a standardised format so NB4 can run the same LLMs on HealthRelease
and compare cross-dataset generalisation.

In [ ]:
os.makedirs('predictions', exist_ok=True)

for name, res in all_llm_results.items():
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')
    out_path  = f'predictions/{safe_name}_predictions.csv'

    pred_df = pd.DataFrame({
        'text'     : test_df['text'].values,
        'label'    : res['y_true'],
        'pred'     : res['y_pred'],
        'pred_raw' : res['preds_raw'],
        'correct'  : (res['y_true'] == res['y_pred']).astype(int),
        'model'    : name,
        'dataset'  : 'HealthStory',
    })
    pred_df.to_csv(out_path, index=False)
    print(f'  ✅ {name} → {out_path}')

print()
print('All prediction files ready for NB4 cross-dataset experiments.')

---
## Section 14 — Summary & Next Steps

In [ ]:
print('=' * 70)
print('SUMMARY — Notebook 3 — LLM Evaluation on FakeHealth HealthStory')
print('=' * 70)

if all_llm_results:
    best_overall = max(all_llm_results, key=lambda k: all_llm_results[k]['f1_macro'])
    best         = all_llm_results[best_overall]
    print(f'\nBest LLM: {best_overall}')
    print(f'  F1-macro : {best["f1_macro"]:.3f}')
    print(f'  F1-Fake  : {best["f1_fake"]:.3f}  (misinformation detection)')
    print(f'  Accuracy : {best["accuracy"]:.3f}')

print()
print('Generated files:')
outputs_list = [
    'Table2_llm_vs_baseline_results.csv     → Table 2 of the report',
    'predictions/<model>_predictions.csv    → Per-prediction files for NB4',
    'fig_llm_confusion_matrices.png         → Confusion matrices',
    'fig_llm_vs_baseline_comparison.png     → Full comparison chart',
]
for o in outputs_list:
    print(f'  • {o}')

print()
print('Next steps (NB4):')
next_steps = [
    'Replicate this notebook on HealthRelease (cross-dataset generalisation)',
    'Fine-tune BERT / RoBERTa on the training set — expected F1 ≈ 0.85–0.92',
    'Add structured health-claim features (criteria S1–S10) to the prompt',
    'Analyse prompt sensitivity: does rephrasing the system prompt affect results?',
    'Run on Monant dataset (after Zenodo access)',
    'Build an ensemble: TF-IDF + best LLM soft-vote',
]
for i, s in enumerate(next_steps, 1):
    print(f'  {i}. {s}')